In [1]:
import os
import json
import warnings
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
import networkx as nx

warnings.filterwarnings("ignore")

# =========================
# Settings (여기만 바꿔서 실행)
# =========================
DATA_PATH = "./training_data_normalized.csv"   # 사용자 파일명
OUT_DIR = "./dag_out/GOLEM"

MODE = "nv"          # "nv"(GOLEM-NV) or "ev"(GOLEM-EV)
INIT_EV = True       # mode="nv"일 때 EV warm start 권장

LAMBDA1 = 0.02       # sparsity
LAMBDA2 = 5.0        # DAG penalty
OMEGA = 0.1          # postprocess threshold ω

MAX_STEPS = 4000
LR = 1e-3
PRINT_EVERY = 400

INIT_EV_STEPS = 1500
RANDOM_STATE = 42
MAX_FEATURES = None

TARGET_CANDIDATES = ["label", "target", "y", "failure", "bank_failure", "default", "is_failed"]


# -------------------------
# expm
# -------------------------
def _expm(A: np.ndarray) -> np.ndarray:
    try:
        from scipy.linalg import expm
        return expm(A)
    except Exception:
        w, V = np.linalg.eig(A)
        Vinv = np.linalg.inv(V)
        return (V @ np.diag(np.exp(w)) @ Vinv).real


# -------------------------
# Data
# -------------------------
def load_numeric_X(
    data_path: str,
    drop_target_candidates: bool = True,
    max_features: Optional[int] = None,
    random_state: int = 42,
) -> Tuple[pd.DataFrame, np.ndarray, List[str]]:
    np.random.seed(random_state)
    df = pd.read_csv(data_path, low_memory=False)

    # 흔한 인덱스 컬럼 제거
    for c in ["Unnamed: 0", "index", "__index_level_0__"]:
        if c in df.columns:
            df = df.drop(columns=[c])

    if drop_target_candidates:
        norm_cols = {c.strip().lower(): c for c in df.columns}
        drop_cols = []
        for tc in TARGET_CANDIDATES:
            if tc in norm_cols:
                drop_cols.append(norm_cols[tc])
        if drop_cols:
            print(f"[INFO] drop target candidates: {drop_cols}")
            df = df.drop(columns=drop_cols)

    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    df = df[num_cols].copy()

    if max_features is not None and df.shape[1] > max_features:
        df = df.iloc[:, :max_features].copy()
        print(f"[INFO] feature capped: {max_features}")

    for c in df.columns:
        if df[c].isna().any():
            df[c] = df[c].fillna(df[c].median())

    col_names = df.columns.tolist()
    X = df.values.astype(float)
    X = X - X.mean(axis=0, keepdims=True)

    print(f"[INFO] X shape: {X.shape}")
    return df, X, col_names


# -------------------------
# h(B) and grad
# -------------------------
def dagness_h_and_grad(B: np.ndarray) -> Tuple[float, np.ndarray]:
    d = B.shape[0]
    M = B * B
    E = _expm(M)
    h = float(np.trace(E) - d)
    grad = (E.T) * (2.0 * B)
    return h, grad


# -------------------------
# LogDet and grad
# -------------------------
def logdet_term_and_grad(B: np.ndarray, eps: float = 1e-12) -> Tuple[float, np.ndarray]:
    d = B.shape[0]
    I = np.eye(d)
    M = I - B

    sign, logabsdet = np.linalg.slogdet(M)
    if sign == 0:
        M = M + eps * np.eye(d)
        sign, logabsdet = np.linalg.slogdet(M)

    term = -float(logabsdet)
    grad = np.linalg.inv(M).T
    return term, grad


# -------------------------
# Likelihoods
# -------------------------
def likelihood_nv_and_grad(X: np.ndarray, B: np.ndarray, eps: float = 1e-12) -> Tuple[float, np.ndarray]:
    n, d = X.shape
    R = X - X @ B
    rss = np.sum(R * R, axis=0) + eps  # (d,)

    L_data = 0.5 * float(np.sum(np.log(rss)))

    XtR = X.T @ R
    grad_data = -(XtR * (1.0 / rss.reshape(1, d)))

    logdet_term, grad_logdet = logdet_term_and_grad(B)
    L = L_data + logdet_term
    grad = grad_data + grad_logdet
    return L, grad


def likelihood_ev_and_grad(X: np.ndarray, B: np.ndarray, eps: float = 1e-12) -> Tuple[float, np.ndarray]:
    n, d = X.shape
    R = X - X @ B
    rss_total = float(np.sum(R * R) + eps)

    L_data = (d / 2.0) * float(np.log(rss_total))

    XtR = X.T @ R
    grad_data = -(d / rss_total) * XtR

    logdet_term, grad_logdet = logdet_term_and_grad(B)
    L = L_data + logdet_term
    grad = grad_data + grad_logdet
    return L, grad


# -------------------------
# Adam
# -------------------------
def adam_optimize(
    X: np.ndarray,
    B_init: np.ndarray,
    mode: str,
    lambda1: float,
    lambda2: float,
    max_steps: int,
    lr: float,
    beta1: float = 0.9,
    beta2: float = 0.999,
    eps: float = 1e-8,
    print_every: int = 200,
) -> np.ndarray:
    B = B_init.copy()
    d = B.shape[0]
    m = np.zeros_like(B)
    v = np.zeros_like(B)

    for t in range(1, max_steps + 1):
        np.fill_diagonal(B, 0.0)

        if mode == "nv":
            L, grad_L = likelihood_nv_and_grad(X, B)
        elif mode == "ev":
            L, grad_L = likelihood_ev_and_grad(X, B)
        else:
            raise ValueError("mode must be 'nv' or 'ev'")

        h, grad_h = dagness_h_and_grad(B)

        grad_l1 = np.sign(B)  # subgradient
        obj = L + lambda1 * float(np.sum(np.abs(B))) + lambda2 * h
        grad = grad_L + lambda1 * grad_l1 + lambda2 * grad_h

        m = beta1 * m + (1 - beta1) * grad
        v = beta2 * v + (1 - beta2) * (grad * grad)
        m_hat = m / (1 - beta1 ** t)
        v_hat = v / (1 - beta2 ** t)

        B = B - lr * m_hat / (np.sqrt(v_hat) + eps)

        if print_every > 0 and (t == 1 or t % print_every == 0 or t == max_steps):
            nnz = int((np.abs(B) > 1e-6).sum())
            print(f"[OPT] step={t:5d} obj={obj:.6f} L={L:.6f} h={h:.6e} nnz~={nnz}")

    np.fill_diagonal(B, 0.0)
    return B


# -------------------------
# Postprocess: threshold + cycle-break
# -------------------------
def break_cycles_by_removing_small_edges(W: np.ndarray) -> np.ndarray:
    W2 = W.copy()
    d = W2.shape[0]

    def build_adj():
        G = {i: [] for i in range(d)}
        for i in range(d):
            for j in range(d):
                if i != j and abs(W2[i, j]) > 0:
                    G[i].append(j)
        return G

    def find_cycle_edges(G):
        color = [0] * d
        parent = [-1] * d

        def dfs(u):
            color[u] = 1
            for v in G[u]:
                if color[v] == 0:
                    parent[v] = u
                    cyc = dfs(v)
                    if cyc is not None:
                        return cyc
                elif color[v] == 1:
                    nodes = [v]
                    cur = u
                    while cur != v and cur != -1:
                        nodes.append(cur)
                        cur = parent[cur]
                    nodes.append(v)
                    nodes = nodes[::-1]
                    return [(a, b) for a, b in zip(nodes[:-1], nodes[1:])]
            color[u] = 2
            return None

        for s in range(d):
            if color[s] == 0:
                cyc = dfs(s)
                if cyc is not None:
                    return cyc
        return None

    removed = 0
    while True:
        cyc = find_cycle_edges(build_adj())
        if cyc is None:
            break
        mags = [(abs(W2[i, j]), i, j) for (i, j) in cyc]
        mags.sort(key=lambda x: x[0])
        _, i_min, j_min = mags[0]
        W2[i_min, j_min] = 0.0
        removed += 1

    if removed:
        print(f"[INFO] cycle-break removed edges: {removed}")
    return W2


# -------------------------
# Save artifacts
# -------------------------
def save_artifacts(W: np.ndarray, col_names: List[str], out_dir: str, alg_name: str) -> None:
    os.makedirs(out_dir, exist_ok=True)

    edges = []
    for i, s in enumerate(col_names):
        for j, t in enumerate(col_names):
            if i != j and abs(W[i, j]) > 0:
                edges.append([s, t, float(W[i, j])])

    edge_df = pd.DataFrame(edges, columns=["source", "target", "weight"])
    edge_df.to_csv(os.path.join(out_dir, f"edges_{alg_name}.csv"), index=False)

    pd.DataFrame(W, index=col_names, columns=col_names).to_csv(
        os.path.join(out_dir, f"adj_{alg_name}.csv")
    )

    G = nx.DiGraph()
    G.add_nodes_from(col_names)
    for _, r in edge_df.iterrows():
        G.add_edge(r["source"], r["target"], weight=float(r["weight"]))

    nx.write_graphml(G, os.path.join(out_dir, f"graph_{alg_name}.graphml"))
    nx.write_gexf(G, os.path.join(out_dir, f"graph_{alg_name}.gexf"))

    with open(os.path.join(out_dir, f"graph_{alg_name}.json"), "w", encoding="utf-8") as f:
        json.dump(
            {
                "nodes": [{"id": n} for n in G.nodes()],
                "edges": [
                    {"source": u, "target": v, "weight": float(G[u][v]["weight"])}
                    for u, v in G.edges()
                ],
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    print(f"[SAVE] {alg_name}")
    print(f"  - {os.path.join(out_dir, f'edges_{alg_name}.csv')} (n_edges={len(edge_df)})")
    print(f"  - {os.path.join(out_dir, f'adj_{alg_name}.csv')}")
    print(f"  - {os.path.join(out_dir, f'graph_{alg_name}.graphml')}")
    print(f"  - {os.path.join(out_dir, f'graph_{alg_name}.gexf')}")
    print(f"  - {os.path.join(out_dir, f'graph_{alg_name}.json')}")


# =========================
# Run
# =========================
np.random.seed(RANDOM_STATE)

_, X, col_names = load_numeric_X(
    DATA_PATH,
    drop_target_candidates=True,
    max_features=MAX_FEATURES,
    random_state=RANDOM_STATE,
)

n, d = X.shape
B0 = np.zeros((d, d), dtype=float)

# Warm start EV -> NV (recommended)
if MODE == "nv" and INIT_EV:
    print("[INFO] Warm start: GOLEM-EV for initialization (recommended for NV).")
    B_ev = adam_optimize(
        X=X,
        B_init=B0,
        mode="ev",
        lambda1=LAMBDA1,
        lambda2=LAMBDA2,
        max_steps=INIT_EV_STEPS,
        lr=LR,
        print_every=PRINT_EVERY,
    )
    B_init = B_ev
else:
    B_init = B0

print(f"[RUN] GOLEM mode={MODE} lambda1={LAMBDA1} lambda2={LAMBDA2} steps={MAX_STEPS} lr={LR}")
B_hat = adam_optimize(
    X=X,
    B_init=B_init,
    mode=MODE,
    lambda1=LAMBDA1,
    lambda2=LAMBDA2,
    max_steps=MAX_STEPS,
    lr=LR,
    print_every=PRINT_EVERY,
)

W = B_hat.copy()
np.fill_diagonal(W, 0.0)

# Post-processing: threshold ω (paper), then ensure DAG
if OMEGA and OMEGA > 0:
    W[np.abs(W) < OMEGA] = 0.0

W = break_cycles_by_removing_small_edges(W)

ALG_NAME = "GOLEM" if MODE == "nv" else "GOLEM_EV"
save_artifacts(W, col_names, OUT_DIR, ALG_NAME)
print("[DONE] GOLEM artifacts saved.")


[INFO] drop target candidates: ['label']
[INFO] X shape: (17881, 13)
[INFO] Warm start: GOLEM-EV for initialization (recommended for NV).
[OPT] step=    1 obj=80.316882 L=80.316882 h=0.000000e+00 nnz~=156
[OPT] step=  400 obj=77.822050 L=77.456679 h=2.841979e-02 nnz~=169
[OPT] step=  800 obj=77.698598 L=77.208329 h=3.744635e-02 nnz~=168
[OPT] step= 1200 obj=77.589648 L=77.105936 h=3.056761e-02 nnz~=169
[OPT] step= 1500 obj=77.502850 L=77.035846 h=2.443534e-02 nnz~=169
[RUN] GOLEM mode=nv lambda1=0.02 lambda2=5.0 steps=4000 lr=0.001
[OPT] step=    1 obj=60.429204 L=59.962246 h=2.441763e-02 nnz~=169
[OPT] step=  400 obj=59.859091 L=59.604126 h=1.146673e-03 nnz~=169
[OPT] step=  800 obj=59.820359 L=59.583944 h=5.164947e-04 nnz~=169
[OPT] step= 1200 obj=59.813922 L=59.582513 h=1.899392e-04 nnz~=168
[OPT] step= 1600 obj=59.813917 L=59.582415 h=2.118298e-04 nnz~=169
[OPT] step= 2000 obj=59.813928 L=59.582513 h=2.151989e-04 nnz~=169
[OPT] step= 2400 obj=59.813935 L=59.582469 h=2.130002e-04 nn